# Czysty notatnik do generowania działającego (miejmy nadzieję) modelu detekcji zagrożeń na podstawie logów Wazuh
## ✨Organicznie uprawiany kod✨
Zalecam korzystanie z menedżera środowisk python `uv`, bo jest lepszy i nie tak ociężały jak `conda` czy domyślny `venv`.
Tworzenie środowiska
```
uv venv .venv --python cp3.14 # argument --python opcjonalnie cp3.14 po prostu jest szybki ;)
```
Aktywacja środowiska
```
source .venv/bin/activate     # to pewnie będzie inaczej na Windows
```
Instalacja bibliotek
```
uv pip install ipykernel pandas numpy
```

## Połączony potok: okna 10 s + historia 60 s (Plik 1) na czystych cechach bez wycieków (Plik 2)

1. **Filtracja źródeł**: wczytujemy wyłącznie alerty Wazuh (`*_wazuh.json`), odrzucając wpisy Suricaty oraz AMinera.
2. **Ekstrakcja czystych cech per zdarzenie (bez reguł, IP i kalendarza)**:
   - anonimizacja adresów IP (`srcip`/`dstip` $\rightarrow$ klasa sieci prywatna/zewnętrzna; surowy `srcip` służy wyłącznie do policzenia unikalnych źródeł w oknie 10 s i jest usuwany),
   - wyznaczenie roli hosta (`WWW`, `mail`, `dns`, `auth`, `other`) z `location` oraz kategorii portu z `data.dstport`,
   - metryki strukturalne i flagi `data.url` (entropia, długość, głębokość ścieżki, zapytania, flagi podatności),
   - maskowanie IP, domen (`.biz`, `.com` itd.), nazw plików i znaczników czasu w `full_log` oraz wyszukiwanie wzorców narzędzi (`nmap`, `dirb`, `wpscan` itp.) i słów kluczowych bez rozróżniania wielkości liter (`lowercase`).
3. **Agregacja w okna 10 s i odrzucenie okien granicznych**: grupowanie zdarzeń w przedziały `10s`, wykluczenie okien przecinających granice z `labels.csv` ze zbioru uczącego przy jednoczesnym zachowaniu ich w pełnej osi czasu.
4. **Sekwencja 60 s (6 kroków × 10 s)**: budowa tensora `(N, 6, F)` z transformacją `log1p` dla liczników.

definicje ekstraktorów czystych cech i wczytanie przedziałów labels.csv

In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
import csv
import ipaddress
import json
import math
import re
import numpy as np
import pandas as pd

# Ścieżka do danych (dopasuj do katalogu roboczego notatnika: "../data/ait_ads" lub "data/ait_ads")
DATA_DIR = Path("../data/ait_ads") if Path("../data/ait_ads").is_dir() else Path("data/ait_ads")

SCENARIOS = ["fox", "harrison", "russellmitchell", "santos", "shaw", "wardbeck", "wheeler", "wilson"]
TRAIN_SCENARIOS = ["fox", "russellmitchell", "shaw", "wardbeck", "wheeler", "wilson", "santos"]
TEST_SCENARIOS = ["harrison"]

# Mapowanie 10 faz z labels.csv na 7 głównych klas ataków (zgodnie z wymaganiami prowadzącego)
ATTACK_7_MAP = {
    "dirb": "dirb",
    "wpscan": "wpscan",
    "dnsteal": "dnsteal",
    "cracking": "cracking",
    "service_scans": "scanning",
    "network_scans": "scanning",
    "webshell": "exploitation",
    "reverse_shell": "exploitation",
    "privilege_escalation": "privilege_escalation",
    "service_stop": "privilege_escalation",
}

# Prekompilowane wyrażenia regularne (case-insensitive)
RX_IPV4 = re.compile(r"\b\d{1,3}(?:\.\d{1,3}){3}\b")
RX_IPV6 = re.compile(r"\b[0-9a-f]{1,4}(?::[0-9a-f]{1,4}){2,7}\b")
RX_DOMAIN = re.compile(r"\b(?:[a-z0-9-]+\.)+(?:biz|com|net|org|io|info|local|lan|intranet)\b")
RX_FILE = re.compile(r"\b[a-z0-9_.-]+\.(?:php|txt|zip|sql|sh|py|exe|bak|tar|gz|log)\b")
RX_TS = re.compile(r"\b\d{4}-\d{2}-\d{2}[t\s]\d{2}:\d{2}:\d{2}(?:\.\d+)?(?:z|[+-]\d{2}:?\d{2})?\b")
RX_TOKEN = re.compile(r"[a-z0-9+/=%_.-]{8,}")
RX_ALNUM = re.compile(r"[a-z0-9]+")

URL_WP_NEEDLES = ("wp-admin", "wp-content", "wp-json", "wp-login", "xmlrpc", "wp-config", "wpscan")
URL_ADMIN_DB_NEEDLES = ("phpmyadmin", "admin", "config", "secret", "pass", "user", "backup", "dump", ".git", "sql")
URL_EXEC_NEEDLES = ("shell", "cmd", "remote", "proc", "include", ".php", ".zip", ".txt", "ocs/")
UA_SCANNER_NEEDLES = ("dirb", "wpscan", "nmap", "ncat", "wget", "headlesschrome")
LOG_AUTH_FAIL_NEEDLES = ("failed", "invalid", "authentication failure", "denied", "unauthorized")
LOG_PRIV_NEEDLES = ("sudo", "su:", "session opened", "chmod", "chown")

_ip_cache: dict[str, int] = {}

def classify_ip(ip_val: object) -> int:
    """Zwraca: 1 = prywatne (wew), 0 = publiczne (zew), -1 = brak/niepoprawne."""
    if not isinstance(ip_val, str) or not ip_val:
        return -1
    clean = ip_val.split(":")[0].strip()
    if clean in _ip_cache:
        return _ip_cache[clean]
    try:
        res = int(ipaddress.ip_address(clean).is_private)
    except ValueError:
        res = -1
    _ip_cache[clean] = res
    return res

def extract_host_role(loc: object) -> str:
    if not isinstance(loc, str):
        return "other"
    l = loc.lower()
    if any(k in l for k in ("apache", "nginx", "access.log", "error.log", "www", "http")):
        return "WWW"
    if any(k in l for k in ("mail", "postfix", "dovecot", "exim", "smtp", "imap")):
        return "mail"
    if any(k in l for k in ("bind", "named", "dns", "unbound")):
        return "dns"
    if any(k in l for k in ("auth", "secure", "sshd", "audit")):
        return "auth"
    return "other"

def port_category(port_val: object) -> str:
    try:
        if not port_val is None:
            print(port_val)
        port = int(float(str(port_val)))
    except (ValueError, TypeError):
        return "other"
    return {80: "http", 443: "https", 53: "dns", 587: "smtp", 5800: "vnc"}.get(port, "other")

def entropy_of(text: str) -> float:
    if not text:
        return 0.0
    counts = Counter(text)
    total = len(text)
    return round(-sum((c / total) * math.log2(c / total) for c in counts.values()), 4)

def mask_and_lower_log(raw: object) -> str:
    if not isinstance(raw, str):
        return ""
    s = raw[:400].lower()  # wszystkie litery są sobie równoważne
    s = RX_IPV4.sub(" ip_token ", s)
    s = RX_IPV6.sub(" ipv6_token ", s)
    s = RX_DOMAIN.sub(" domain_token ", s)
    s = RX_FILE.sub(" file_token ", s)
    s = RX_TS.sub(" ts_token ", s)
    return s

def load_intervals(path: Path) -> dict[str, list[tuple[float, float, str]]]:
    result: dict[str, list[tuple[float, float, str]]] = defaultdict(list)
    with path.open(newline="", encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            sc = row["scenario"].strip().lower()
            start, end = float(row["start"]), float(row["end"])
            raw_attack = row["attack"].strip()
            mapped_attack = ATTACK_7_MAP.get(raw_attack, raw_attack)
            result[sc].append((start, end, mapped_attack))
    for sc in result:
        result[sc].sort()
    return result

intervals = load_intervals(DATA_DIR / "labels.csv")
print("Wczytano przedziały z labels.csv dla scenariuszy:", list(intervals.keys()))

wczytanie logów Wazuh, ekstrakcja czystych cech i etykietowanie czasu

In [38]:
RX_DNS_LOG = re.compile(r"(?:query|qname|rrname)[\"'=:\s]+([a-z0-9._-]{3,253})", re.I)

def extract_extra(alert, data, url_raw):
    raw = str(alert.get("full_log") or "")
    dns_dict = data.get("dns") if isinstance(data.get("dns"), dict) else {}

    q = data.get("query") or dns_dict.get("rrname")
    if not q:
        m = RX_DNS_LOG.search(raw)
        q = m.group(1) if m else ""
    q = str(q or "").strip(".").lower()
    q_labels = q.split(".") if q else []

    qtype = str(data.get("query_type") or dns_dict.get("type") or "").upper()
    rcode = str(data.get("rcode") or dns_dict.get("rcode") or "").upper()
    dstport_num = pd.to_numeric(data.get("dstport"), errors="coerce")

    return {
        "srcip_raw": str(data.get("srcip") or "").strip(), # za mało danych do ekstrakcji
        "dstip_raw": str(data.get("dstip") or "").strip(), # za mało danych do ekstrakcji
#        "dstport_num": dstport_num,  # za mało danych do ekstrakcji
        "has_url": int(bool(url_raw)),
        "is_dns": int(bool(q) or dstport_num == 53),
        # Nazwa domeny używana tylko tymczasowo do liczenia unikalnych nazw; usuwana przed modelem
        "dns_query": q,
#        "dns_name_len": float(len(q)) if q else np.nan, # za mało danych do ekstrakcji
#        "dns_entropy": entropy_of(q) if q else np.nan, # za mało danych do ekstrakcji
        "dns_depth": float(max(len(q_labels) - 2, 0)) if q else np.nan,
#        "dns_is_txt": int(qtype in {"TXT", "NULL"}), # za mało danych do ekstrakcji
#        "dns_nxdomain": int(rcode == "NXDOMAIN" or "nxdomain" in raw.lower()), # za mało danych do ekstrakcji
    }

rows = []
for sc in SCENARIOS:
    path = DATA_DIR / f"{sc}_wazuh.json"
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            alert = json.loads(line)

            # 1. Odrzucenie Suricaty i AMinera (tylko czysty Wazuh)
            location = str(alert.get("location") or "")
            rule = alert.get("rule") or {}
            decoder = alert.get("decoder") or {}
            rule_desc = str(rule.get("description") or "").lower()
            rule_groups = [str(g).lower() for g in (rule.get("groups") or [])]
            decoder_name = str(decoder.get("name") or "").lower()

            if (
                "suricata" in location.lower()
#                or "suricata" in rule_desc
#                or "suricata" in rule_groups
#                or "aminer" in location.lower()
#                or "aminer" in decoder_name
#                or "aminer" in rule_groups
            ):
                continue

            data = alert.get("data") or {}
            srcip_raw = str(data.get("srcip") or "").strip()
            dstip_raw = str(data.get("dstip") or "").strip()
            dst_potr = data.get("dstport")
            src_cls = classify_ip(srcip_raw)
            dst_cls = classify_ip(dstip_raw)

            url_raw = str(data.get("url") or "")
            url_low = url_raw.lower()
            log_masked = mask_and_lower_log(alert.get("full_log"))

            dstuser = str(data.get("dstuser") or "").strip()
            srcuser = str(data.get("srcuser") or "").strip()
            proto = str(data.get("protocol") or "").upper()

            rows.append({
                "scenario": sc,
                "timestamp": alert.get("@timestamp"),
                # Tymczasowe pole tylko do zliczenia unikalnych źródeł w oknie 10s (nie trafia do modelu)
                "srcip_anon": srcip_raw,
                # Zanonimizowane cechy sieciowe i roli hosta
                "src_private": int(src_cls == 1),
#                "src_external": int(src_cls == 0), # wyłącznie jedna wartość + wygląda jak przeciwieństwo src_private - redundancja
                "dst_private": int(dst_cls == 1),
#                "dst_external": int(dst_cls == 0), # wyłącznie jedna wartość + wygląda jak przeciwieństwo dst_private - redundancja
                "host_role": extract_host_role(location),
#                "dstport_cat": port_category(data.get("dstport")), # za mało danych do ekstrakcji
                # Tożsamość i protokół (bez nazw użytkowników)
                "is_root": int(dstuser == "root"),
                "has_user": int(bool(dstuser or srcuser)),
                "is_get": int(proto == "GET"),
                "is_post": int(proto == "POST"),
                "is_head": int(proto == "HEAD"),
                # Cechy strukturalne URL
                "url_len": float(len(url_raw)),
                "url_entropy": entropy_of(url_low),
                "url_parts": float(len(url_low.split("/"))) if url_low else 0.0,
                "url_query": int("?" in url_low),
                "url_encoded": float(url_low.count("%")),
                "url_max_token": float(max((len(t) for t in RX_TOKEN.findall(url_low)), default=0)),
                # Flagi wzorców URL oraz narzędzi skanujących (z zamaskowanego logu lowercase)
                "flag_wp": int(any(n in url_low for n in URL_WP_NEEDLES)),
                "flag_admin_db": int(any(n in url_low for n in URL_ADMIN_DB_NEEDLES)),
                "flag_exec_file": int(any(n in url_low for n in URL_EXEC_NEEDLES)),
                "flag_ua_scanner": int(any(n in log_masked for n in UA_SCANNER_NEEDLES)),
                # Cechy leksykalne z zamaskowanego full_log (lowercase)
                "log_auth_fail": int(any(n in log_masked for n in LOG_AUTH_FAIL_NEEDLES)),
                "log_priv_cmd": int(any(n in log_masked for n in LOG_PRIV_NEEDLES)),
                "log_has_domain": int("domain_token" in log_masked),
                "log_has_file": int("file_token" in log_masked),
                **extract_extra(alert, data, url_raw),
            })

df_events = pd.DataFrame(rows)
df_events["event_time"] = pd.to_datetime(df_events["timestamp"], utc=True)
times = ((df_events["event_time"] - pd.Timestamp("1970-01-01", tz="UTC")) / pd.Timedelta(seconds=1)).to_numpy()

# Przypisanie etykiet czasowych z labels.csv w obrębie każdego scenariusza
scenarios_arr = df_events["scenario"].to_numpy()
labels = np.full(len(df_events), "background", dtype=object)
match_count = np.zeros(len(df_events), dtype=np.uint8)

for sc, phase_intervals in intervals.items():
    sc_mask = scenarios_arr == sc
    for start, end, phase in phase_intervals:
        in_phase = sc_mask & (times >= start) & (times < end)
        match_count[in_phase] += 1
        labels[in_phase] = phase

assert not (match_count > 1).any(), "Nachodzące etykiety — sprawdź labels.csv"
df_events["attack_phase"] = labels
df_events = df_events.drop(columns=["timestamp"])

print("Wczytano czystych zdarzeń Wazuh:", len(df_events))
display(pd.crosstab(df_events["scenario"], df_events["attack_phase"]))

Wczytano czystych zdarzeń Wazuh: 1986905


attack_phase,background,cracking,dirb,dnsteal,exploitation,privilege_escalation,scanning,wpscan
scenario,,,,,,,,
fox,28801,174,406075,347,0,10,199,4749
harrison,31483,349,410697,1370,18,37,181,4824
russellmitchell,15005,8,4459,383,0,6,87,3168
santos,29550,389,4459,543,12,22,44,3270
shaw,29581,239,4459,105,3,19,4,729
wardbeck,33529,384,4459,9,15,13,25,750
wheeler,47037,0,412693,175,0,9,84,6638
wilson,66435,486,423894,1101,9,22,81,3199


In [40]:
def print_unique_values(df):
    for col in df.columns:
        try:
            counts = df[col].value_counts(dropna=False)
        except TypeError:  # lists or dicts aren't hashable
            counts = df[col].astype(str).value_counts(dropna=False)

        print(f"\n=== {col} ({counts.shape[0]} unique) ===")
        print(counts)

print_unique_values(ev_win)


=== alert_count (484 unique) ===
alert_count
3       36905
6       12018
9        3413
12       3278
1        2728
        ...  
3753        1
3276        1
1796        1
1382        1
1802        1
Name: count, Length: 484, dtype: int64

=== src_private_cnt (468 unique) ===
src_private_cnt
0       61456
1         676
4         211
2         174
3         155
        ...  
3750        1
3273        1
1796        1
1379        1
1799        1
Name: count, Length: 468, dtype: int64

=== dst_private_cnt (3 unique) ===
dst_private_cnt
0    62886
3      374
6        2
Name: count, dtype: int64

=== root_user_cnt (6 unique) ===
root_user_cnt
0    63238
3       15
7        4
1        2
4        2
6        1
Name: count, dtype: int64

=== known_user_cnt (8 unique) ===
known_user_cnt
0    62408
1      280
3      219
4      184
2      122
5       36
6        9
7        4
Name: count, dtype: int64

=== http_get_cnt (454 unique) ===
http_get_cnt
0       62627
2          78
4          29
8        

szkielet okien 10 s z Pliku 1 + odrzucenie okien granicznych

In [ ]:
WINDOW_SECONDS = 10

df_events["window_start"] = df_events["event_time"].dt.floor("10s")
keys = ["scenario", "window_start"]

# Etap 1: statystyki na źródło w oknie
src_ev = df_events[df_events["srcip_raw"] != ""]
per_src = src_ev.groupby(keys + ["srcip_raw"]).agg(
    src_alerts=("attack_phase", "size"),
#    src_unique_dstports=("dstport_num", "nunique"),
    src_unique_dstips=("dstip_raw", "nunique"),
    src_dns_q=("is_dns", "sum"),
    src_dns_unique=("dns_query", lambda s: s[s != ""].nunique()),
#    src_dns_txt=("dns_is_txt", "sum"),
#    src_dns_nxdomain=("dns_nxdomain", "sum"),
    src_auth_fail=("log_auth_fail", "sum"),
).reset_index()

# Etap 2: agregacja źródeł do okna (maksima i liczby)
src_win = per_src.groupby(keys).agg(
    n_unique_sources=("srcip_raw", "size"),
    max_src_alerts=("src_alerts", "max"),
#    max_src_unique_dstports=("src_unique_dstports", "max"),
    max_src_unique_dstips=("src_unique_dstips", "max"),
    max_src_dns_q=("src_dns_q", "max"),
    max_src_dns_unique=("src_dns_unique", "max"),
#    max_src_dns_txt=("src_dns_txt", "max"),
    max_src_auth_fail=("src_auth_fail", "max"),
    sources_with_dns=("src_dns_q", lambda s: int((s > 0).sum())),
)

# Zdarzeniowe liczniki okna
ev_win = df_events.groupby(keys).agg(
    alert_count=("attack_phase", "size"),
    src_private_cnt=("src_private", "sum"),
#    src_external_cnt=("src_external", "sum"),
    dst_private_cnt=("dst_private", "sum"),
#    dst_external_cnt=("dst_external", "sum"),
    root_user_cnt=("is_root", "sum"),
    known_user_cnt=("has_user", "sum"),
    http_get_cnt=("is_get", "sum"),
    http_post_cnt=("is_post", "sum"),
    http_head_cnt=("is_head", "sum"),
    url_query_cnt=("url_query", "sum"),
    flag_wp_cnt=("flag_wp", "sum"),
    flag_admin_db_cnt=("flag_admin_db", "sum"),
    flag_exec_file_cnt=("flag_exec_file", "sum"),
    flag_ua_scanner_cnt=("flag_ua_scanner", "sum"),
    log_auth_fail_cnt=("log_auth_fail", "sum"),
    log_priv_cmd_cnt=("log_priv_cmd", "sum"),
    log_domain_cnt=("log_has_domain", "sum"),
    log_file_cnt=("log_has_file", "sum"),
    dns_cnt=("is_dns", "sum"),
#    dns_txt_cnt=("dns_is_txt", "sum"),
#    dns_nxdomain_cnt=("dns_nxdomain", "sum"),
    label_count=("attack_phase", "nunique"),
    attack_phase=("attack_phase", "first"),
)

# Statystyki URL liczone tylko po zdarzeniach, które mają URL (brak URL nie jest zerem)
url_ev = df_events[df_events["has_url"] == 1]
url_win = url_ev.groupby(keys).agg(
    url_cnt=("has_url", "size"),
    mean_url_len=("url_len", "mean"),
    max_url_len=("url_len", "max"),
    mean_url_entropy=("url_entropy", "mean"),
    max_url_entropy=("url_entropy", "max"),
    mean_url_parts=("url_parts", "mean"),
    max_url_encoded=("url_encoded", "max"),
    max_url_token=("url_max_token", "max"),
)

# Statystyki DNS liczone tylko po zdarzeniach DNS
dns_ev = df_events[df_events["is_dns"] == 1]
dns_win = dns_ev.groupby(keys).agg(
#    mean_dns_name_len=("dns_name_len", "mean"),
#    max_dns_name_len=("dns_name_len", "max"),
#    mean_dns_entropy=("dns_entropy", "mean"),
#    max_dns_entropy=("dns_entropy", "max"),
    max_dns_depth=("dns_depth", "max"),
)

# Rola hosta i kategoria portu jako liczniki
idx = [df_events["scenario"], df_events["window_start"]]
role_win = pd.crosstab(idx, df_events["host_role"]).add_prefix("role_")
#port_win = pd.crosstab(idx, df_events["dstport_cat"]).add_prefix("port_")

all_windows_full = (
    ev_win.join(src_win, how="left")
    .join(url_win, how="left")
    .join(dns_win, how="left")
    .join(role_win, how="left")
#    .join(port_win, how="left")
)

all_windows_full["top_src_share"] = (
    all_windows_full["max_src_alerts"] / all_windows_full["alert_count"]
)

# Braki w cechach numerycznych oznaczają "brak zdarzeń tego typu" i zostają zerami,
# a url_cnt / dns_cnt pozwalają je odróżnić od prawdziwych zer
num_cols = all_windows_full.select_dtypes("number").columns
all_windows_full[num_cols] = all_windows_full[num_cols].fillna(0)

# Usunięcie surowych pól, których nie wolno podać modelowi
df_events = df_events.drop(columns=["srcip_raw", "dstip_raw", "dns_query"]) # "dstport_num"


# 2. Rozbicie ról hostów (WWW, mail, dns, auth, other) i portów na liczniki w oknie 10s
role_counts = df_events.groupby(keys + ["host_role"]).size().unstack(fill_value=0)
role_counts.columns = [f"role_{c}_cnt" for c in role_counts.columns]

#port_counts = df_events.groupby(keys + ["dstport_cat"]).size().unstack(fill_value=0)
#port_counts.columns = [f"port_{c}_cnt" for c in port_counts.columns]

# Zachowujemy pełną tablicę wszystkich okien (również granicznych) do budowy ciągłej historii 60s
#all_windows_full = windows.join(role_counts).join(port_counts)

# 3. Wykluczenie okien przecinających granice faz z labels.csv (dokładnie jak w Pliku 1)
windows_reset = all_windows_full.reset_index()
boundary_crossing = np.zeros(len(windows_reset), dtype=bool)

for sc, phase_intervals in intervals.items():
    mask = windows_reset["scenario"].eq(sc).to_numpy()
    positions = np.flatnonzero(mask)
    starts = (
        (windows_reset.loc[mask, "window_start"] - pd.Timestamp("1970-01-01", tz="UTC"))
        / pd.Timedelta(seconds=1)
    ).to_numpy()

    crosses = np.zeros(len(starts), dtype=bool)
    for attack_start, attack_end, _ in phase_intervals:
        for boundary in (attack_start, attack_end):
            crosses |= (starts < boundary) & (boundary < starts + WINDOW_SECONDS)
    boundary_crossing[positions] = crosses

invalid = boundary_crossing | windows_reset["label_count"].gt(1).to_numpy()
df_windows = windows_reset.loc[~invalid].drop(columns=["label_count"]).reset_index(drop=True)

print(f"Wszystkie okna 10s: {len(windows_reset)} | Odrzucone graniczne: {int(invalid.sum())} | Czyste okna: {len(df_windows)}")
print(pd.crosstab(df_windows["scenario"], df_windows["attack_phase"]))

In [30]:
print(pd.crosstab(df_windows["scenario"], df_windows["attack_phase"]))

attack_phase     background  cracking  dirb  dnsteal  exploitation  \
scenario                                                             
fox                    6616        38   117       77             0   
harrison               6998        77   157      281             3   
russellmitchell        3605         5     1       86             0   
santos                 6389        83     2      104             3   
shaw                   7021        56     1       27             1   
wardbeck               7498        79     2        6             3   
wheeler               10049         0    99       43             0   
wilson                13177        89   120      191             1   

attack_phase     privilege_escalation  scanning  wpscan  
scenario                                                 
fox                                 2        25       4  
harrison                            4        34       6  
russellmitchell                     3         8       2  
santos   

budowa sekwencji 60 s (N, 6, F) i skalowanie log1p

In [ ]:
SEQ_LEN = 6        # 6 kroków × 10 s = 60 sekund kontekstu historycznego
STEP_SECONDS = 10

non_feature_cols = {"scenario", "window_start", "label_count", "attack_phase"}
CONTINUOUS_COLS = {
    "mean_url_len", "max_url_len", "mean_url_entropy", "max_url_entropy", "mean_url_parts",
    "mean_dns_name_len", "max_dns_name_len", "mean_dns_entropy", "max_dns_entropy",
    "max_dns_depth", "top_src_share",
}
sequence_features = [c for c in all_windows_full.columns if c not in non_feature_cols]
log1p_indices = [i for i, c in enumerate(sequence_features) if c not in CONTINUOUS_COLS]

# Zabezpieczenia: żadnych identyfikatorów ani etykiet w cechach
forbidden = {"srcip_raw", "dstip_raw", "dns_query", "label_count", "attack_phase", "scenario"}
assert not (set(sequence_features) & forbidden)

# Pełna oś czasu okien (włącznie z granicznymi, aby historia 60s nie miała sztucznych dziur)
all_windows_matrix = all_windows_full.reindex(columns=sequence_features, fill_value=0).astype("float32")

X_sequences = np.zeros((len(df_windows), SEQ_LEN, len(sequence_features)), dtype=np.float32)
for step in range(SEQ_LEN):
    lag = (SEQ_LEN - 1 - step) * STEP_SECONDS
    requested_index = pd.MultiIndex.from_arrays(
        [df_windows["scenario"], df_windows["window_start"] - pd.Timedelta(seconds=lag)],
        names=["scenario", "window_start"],
    )
    X_sequences[:, step, :] = all_windows_matrix.reindex(
        requested_index, fill_value=0.0
    ).to_numpy(dtype=np.float32)

# Skalowanie log1p dla wszystkich kolumn zliczeniowych
X_sequences[:, :, log1p_indices] = np.log1p(np.maximum(X_sequences[:, :, log1p_indices], 0.0))
assert np.isfinite(X_sequences).all()

# Podział na zbiór treningowy i testowy po scenariuszach
train_mask = df_windows["scenario"].isin(TRAIN_SCENARIOS).to_numpy()
test_mask = df_windows["scenario"].isin(TEST_SCENARIOS).to_numpy()

X_train_seq = X_sequences[train_mask]
y_train = df_windows.loc[train_mask, "attack_phase"].to_numpy()

X_test_seq = X_sequences[test_mask]
y_test = df_windows.loc[test_mask, "attack_phase"].to_numpy()

print(f"Gotowy tensor sekwencji X_train_seq: {X_train_seq.shape} (okna, 6 kroków po 10s, {len(sequence_features)} czystych cech)")
print(f"Gotowy tensor sekwencji X_test_seq : {X_test_seq.shape}")

In [26]:
# Przygotowanie etykiet i standaryzacji
# Etykiety attack_phase trzeba zamienić na liczby całkowite. Skalowanie cech warto dopasować tylko na zbiorze treningowym, bo inaczej statystyki z danych testowych przeciekną do treningu.
import tensorflow as tf
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

train_sc = sorted(TRAIN_SCENARIOS)
VAL_SCENARIOS = train_sc[-2:]
FIT_SCENARIOS = train_sc[:-2]

scen_arr = df_windows["scenario"].to_numpy()
fit_mask = np.isin(scen_arr, FIT_SCENARIOS)
val_mask = np.isin(scen_arr, VAL_SCENARIOS)

X_fit_seq = X_sequences[fit_mask]
y_fit = df_windows.loc[fit_mask, "attack_phase"].to_numpy()
X_val_seq = X_sequences[val_mask]
y_val = df_windows.loc[val_mask, "attack_phase"].to_numpy()

print("Trening:", FIT_SCENARIOS, len(y_fit))
print("Walidacja:", VAL_SCENARIOS, len(y_val))

# Enkoder tylko na treningu
le = LabelEncoder().fit(y_fit)
y_fit_enc = le.transform(y_fit)

# Walidacja: odrzucamy klasy spoza treningu i raportujemy to
known = np.isin(y_val, le.classes_)
print(f"Odrzucone próbki walidacji: {(~known).sum()} z {len(known)}")
print("Odrzucone klasy:", np.unique(y_val[~known]))
X_val_seq = X_val_seq[known]
y_val_enc = le.transform(y_val[known])

# Standaryzacja tylko na treningu; cechy stałe nie są skalowane
flat = X_fit_seq.reshape(-1, X_fit_seq.shape[-1])
mean = flat.mean(axis=0)
std = flat.std(axis=0)
std[std < 1e-8] = 1.0

X_fit_s = ((X_fit_seq - mean) / std).astype(np.float32)
X_val_s = ((X_val_seq - mean) / std).astype(np.float32)

# Wagi klas: łagodne (pierwiastek z wag balanced), tylko na treningu
classes = np.unique(y_fit_enc)
weights = compute_class_weight("balanced", classes=classes, y=y_fit_enc)
weights = np.sqrt(weights)  # łagodzenie: rzadkie klasy nadal ważniejsze, ale bez dominacji
class_weight = dict(zip(classes.tolist(), weights.tolist()))

print({le.classes_[k]: round(v, 3) for k, v in class_weight.items()})



Trening: ['fox', 'russellmitchell', 'santos', 'shaw', 'wardbeck'] 31885
Walidacja: ['wheeler', 'wilson'] 23787
Odrzucone próbki walidacji: 0 z 23787
Odrzucone klasy: []
{'background': 0.358, 'cracking': 3.908, 'dirb': 5.692, 'dnsteal': 3.645, 'exploitation': 23.862, 'privilege_escalation': 19.035, 'scanning': 9.628, 'wpscan': 19.035}


In [ ]:
print(VAL_SCENARIOS)
print(FIT_SCENARIOS)

In [27]:
test_mask = np.isin(scen_arr, TEST_SCENARIOS)

X_test_seq = X_sequences[test_mask]
y_test = df_windows.loc[test_mask, "attack_phase"].to_numpy()

print("Test:", TEST_SCENARIOS, len(y_test))
print(pd.Series(y_test).value_counts())

# Odrzucamy klasy, których nie ma w treningu (enkoder jest tylko z treningu)
known_test = np.isin(y_test, le.classes_)
print(f"Odrzucone próbki testowe: {(~known_test).sum()} z {len(known_test)}")
print("Odrzucone klasy:", np.unique(y_test[~known_test]))

X_test_seq = X_test_seq[known_test]
y_test_enc = le.transform(y_test[known_test])

# Standaryzacja statystykami z treningu (bez ponownego dopasowania)
X_test_s = ((X_test_seq - mean) / std).astype(np.float32)


Test: ['harrison'] 7560
background              6998
dnsteal                  281
dirb                     157
cracking                  77
scanning                  34
wpscan                     6
privilege_escalation       4
exploitation               3
Name: count, dtype: int64
Odrzucone próbki testowe: 0 z 7560
Odrzucone klasy: []


In [34]:
# Architektura i trening
# Model składa się z jednej lub dwóch warstw LSTM i głowy softmax. Walidację najlepiej wydzielić po scenariuszach, a nie losowo po oknach, bo okna sąsiednie są silnie skorelowane.

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(SEQ_LEN, X_fit_s.shape[-1])),
    tf.keras.layers.Masking(mask_value=0.0),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(len(le.classes_), activation="softmax"),
])


model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

early = tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

from sklearn.metrics import f1_score

class MacroF1Early(tf.keras.callbacks.Callback):
    def __init__(self, X_val, y_val, patience=10):
        super().__init__()
        self.X_val, self.y_val, self.patience = X_val, y_val, patience
        self.best, self.best_w, self.wait = -1.0, None, 0

    def on_epoch_end(self, epoch, logs=None):
        pred = np.argmax(self.model.predict(self.X_val, verbose=0), axis=1)
        f1 = f1_score(self.y_val, pred, average="macro", zero_division=0)
        print(f" val_macro_f1: {f1:.4f}")
        if f1 > self.best + 1e-3:
            self.best, self.best_w, self.wait = f1, self.model.get_weights(), 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True

    def on_train_end(self, logs=None):
        if self.best_w is not None:
            self.model.set_weights(self.best_w)

history = model.fit(
    X_fit_s, y_fit_enc,
    validation_data=(X_val_s, y_val_enc),
    epochs=50,
    batch_size=64,
    class_weight=class_weight,
    callbacks=[MacroF1Early(X_val_s, y_val_enc)],
    shuffle=True,
)



Epoch 1/50
497/499 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9631 - loss: 0.3902 val_macro_f1: 0.2988
499/499 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.9631 - loss: 0.3903 - val_accuracy: 0.9846 - val_loss: 0.3636
Epoch 2/50
496/499 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9797 - loss: 0.3135 val_macro_f1: 0.3844
499/499 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9796 - loss: 0.3139 - val_accuracy: 0.9850 - val_loss: 0.2334
Epoch 3/50
497/499 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9804 - loss: 0.2987 val_macro_f1: 0.4498
499/499 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.9805 - loss: 0.2986 - val_accuracy: 0.9856 - val_loss: 0.1780
Epoch 4/50
495/499 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9808 - loss: 0.2944 val_macro_f1: 0.4256
499/499 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.9808 - loss: 0.2939 - val_accuracy: 0.9857 - val_loss: 0.2719
Epoch 5/50
493/499 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9808 - loss: 0.2889 val_macro_f1:

In [35]:
# Ewaluacja
# Przy niezbalansowanych fazach ataku dokładność (accuracy) bywa myląca. Lepiej patrzeć na macro-F1 i macierz pomyłek.

from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test_s).argmax(axis=1)

labels_all = np.arange(len(le.classes_))
print(classification_report(
    y_test_enc,
    y_pred,
    labels=labels_all,
    target_names=le.classes_.astype(str),
    zero_division=0,
))
print(confusion_matrix(y_test_enc, y_pred, labels=labels_all))



237/237 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 
                      precision    recall  f1-score   support

          background       0.95      1.00      0.97      6998
            cracking       0.00      0.00      0.00        77
                dirb       1.00      1.00      1.00       157
             dnsteal       0.00      0.00      0.00       281
        exploitation       0.67      0.67      0.67         3
privilege_escalation       1.00      1.00      1.00         4
            scanning       1.00      0.06      0.11        34
              wpscan       0.80      0.67      0.73         6

            accuracy                           0.95      7560
           macro avg       0.68      0.55      0.56      7560
        weighted avg       0.90      0.95      0.92      7560

[[6998    0    0    0    0    0    0    0]
 [  76    0    0    0    1    0    0    0]
 [   0    0  157    0    0    0    0    0]
 [ 281    0    0    0    0    0    0    0]
 [   1    0    0    0    2    0    0   

In [ ]:
print(df_events.groupby("scenario")["is_dns"].sum())
print(df_events.loc[df_events["is_dns"] == 1, ["dns_name_len", "dns_entropy", "dns_depth", "dns_is_txt"]].describe())


In [ ]:
flag_cols = ["flag_wp_cnt", "flag_ua_scanner_cnt", "log_priv_cmd_cnt", "log_auth_fail_cnt"]
print(all_windows_full.groupby("attack_phase")[flag_cols].apply(lambda d: (d > 0).mean()).round(2))


In [ ]:
dns_raw = df_events[df_events["is_dns"] == 1]
print(dns_raw[["dstport_cat", "host_role"]].value_counts().head(10))

# Surowe wartości pól zapytań, jeśli zostały zapisane
print(df_events["dstport_cat"].value_counts().head(10))
print(df_events["host_role"].value_counts())


In [ ]:
print(per_src.groupby("scenario")[["src_unique_dstports", "src_unique_dstips"]].max())


In [ ]:
from collections import Counter

key_counter = Counter()
decoder_counter = Counter()
example = None

for sc in ["fox", "harrison", "wilson"]:
    with (DATA_DIR / f"{sc}_wazuh.json").open(encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 20000:
                break
            a = json.loads(line)
            d = a.get("data") or {}
            key_counter.update(d.keys())
            decoder_counter.update([str((a.get("decoder") or {}).get("name"))])
            if example is None and any("dns" in k.lower() or "port" in k.lower() for k in d):
                example = a

print("Klucze w data:", key_counter.most_common(40))
print("Dekodery:", decoder_counter.most_common(15))
print("Przykładowy alert DNS/port:")
print(json.dumps(example, indent=2, ensure_ascii=False)[:3000] if example else "brak")
